# Chapter 9 -- 词向量 Embedding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter09_embedding.ipynb)

**本章目标：** 理解 Embedding 的原理，学会用 Keras Tokenizer 把文章转换成整数序列，并掌握 padding。

**预计时间：** 45 分钟

> 推荐阅读：[Jay Alammar -- Illustrated Word2Vec](https://jalammar.github.io/illustrated-word2vec/)（配图极好）

## 9.1 TF-IDF 的局限

TF-IDF 很有用，但它有两个根本缺陷：

1. **不懂语义**："good" 和 "great" 在 TF-IDF 里是完全独立的两个词，没有任何关联
2. **高维稀疏**：词汇量 50,000 个词，每篇文章向量有 50,000 维，但 99%+ 都是 0

**Embedding（词嵌入）** 解决这两个问题：
- 每个词映射到一个**低维稠密向量**（例如 128 维）
- 语义相近的词，向量空间里距离也近
- 这些向量是**训练中自动学习**的，不需要手工设计

## 9.2 Word2Vec 的核心直觉

Word2Vec（2013，Google）是 Embedding 的奠基性工作。

核心思想：**一个词的含义，由它周围的词决定。**

例如：
- "I bought a ___ at the store" → 空格里大概率是名词（apple/phone/ticket）
- "The ___ scored a goal" → 大概率是运动员

Word2Vec 强迫模型根据上下文预测中心词（或反过来），在这个过程中，词向量自然地学到了语义。

**神奇的结果：**
```
vec('king') - vec('man') + vec('woman') ≈ vec('queen')
vec('Paris') - vec('France') + vec('Italy') ≈ vec('Rome')
```
这说明向量空间编码了真实的语义关系！

In [ ]:
# 直观演示：用预训练向量感受一下「相似词」
# 注意：gensim 下载需要时间，这里用简化版本演示概念
import numpy as np

# 手工构造一个微型「词向量空间」来演示概念
# 真实 Embedding 有几百维，这里用 2 维方便可视化
words = ['football', 'soccer', 'basketball', 'tennis',
         'economy', 'finance', 'market', 'stock',
         'politics', 'election', 'government', 'vote']

# 假设的 2D 向量（真实情况下由训练自动生成）
vectors = np.array([
    [0.9, 0.1],  # football
    [0.85, 0.15], # soccer
    [0.8, 0.2],  # basketball
    [0.75, 0.25], # tennis
    [0.1, 0.9],  # economy
    [0.15, 0.85], # finance
    [0.2, 0.8],  # market
    [0.25, 0.75], # stock
    [0.1, 0.1],  # politics
    [0.15, 0.1], # election
    [0.2, 0.15], # government
    [0.25, 0.1], # vote
])

# 余弦相似度
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# football 和各词的相似度
football_vec = vectors[0]
print('与 "football" 的余弦相似度：')
for word, vec in zip(words, vectors):
    sim = cosine_sim(football_vec, vec)
    print(f'  {word:12s}: {sim:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue']*4 + ['coral']*4 + ['mediumseagreen']*4
for i, (word, vec) in enumerate(zip(words, vectors)):
    ax.scatter(vec[0], vec[1], color=colors[i], s=100, zorder=3)
    ax.annotate(word, vec, fontsize=9, xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Dimension 1 (Sports ↔ non-Sports)')
ax.set_ylabel('Dimension 2 (Economy ↔ non-Economy)')
ax.set_title('词向量空间（示意图）：语义相近的词聚在一起', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Sports 词聚在右上角，Economy 词聚在左上角，Politics 词在左下角')

## 9.3 Keras Tokenizer：把词转换成整数

在用神经网络之前，我们需要把文字转换成整数序列（每个整数代表词表里的一个词）。

Keras 的 `Tokenizer` 帮我们做这件事：

```
文章："sports team wins championship"
   ↓  Tokenizer
序列：[4, 12, 7, 156]   ← 每个词对应一个整数编号
```

然后 `Embedding` 层把每个整数映射到一个向量：

```
[4, 12, 7, 156]
   ↓  Embedding 层（维度=128）
[[0.2, -0.1, ...],    ← 词 4 的 128 维向量
 [0.8,  0.3, ...],    ← 词 12 的 128 维向量
 ...]
```

整篇文章变成一个矩阵：**序列长度 × 嵌入维度**

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)

le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

print(f'训练集：{len(X_train_text)} 篇，测试集：{len(X_test_text)} 篇')
print(f'类别：{le.classes_}')

In [ ]:
# 建立词表，只保留最高频的 20,000 个词
VOCAB_SIZE = 20000
MAX_LEN    = 100   # 每篇文章截断/补齐到 100 个词
EMBED_DIM  = 128   # 每个词用 128 维向量表示

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)   # 只在训练集上建立词表！

# 转换为整数序列
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

# 打印几个例子
print('原始文章：', X_train_text[0][:80])
print('整数序列：', X_train_seq[0][:20], '...')
print(f'\n词表大小：{len(tokenizer.word_index)} 个不同的词')
print(f'序列长度（前5篇）：{[len(s) for s in X_train_seq[:5]]}')

## 9.4 Padding（补齐/截断）

问题：每篇文章长度不同，神经网络需要固定长度的输入。

解决：`pad_sequences`
- 太短的序列：在**前面**补 0（padding）
- 太长的序列：截断前面，保留后面（`truncating='pre'`）

```
原始：[4, 12, 7, 156]        → 只有 4 个词
补齐：[0, 0, 0, 0, ..., 4, 12, 7, 156]  → 100 个位置
```

补 0 的位置在 Embedding 层中对应零向量，不影响学习。

In [ ]:
# pad_sequences：统一长度为 MAX_LEN=100
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN,
                             padding='pre', truncating='pre')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN,
                             padding='pre', truncating='pre')

print(f'Padding 后训练集形状：{X_train_pad.shape}')   # (N, 100)
print(f'Padding 后测试集形状：{X_test_pad.shape}')
print()
print('第一篇文章（padding 后最后 20 个位置）：')
print(X_train_pad[0, -20:])  # 前面是 0，后面是真实词编号

In [ ]:
# 文章长度分布
import matplotlib.pyplot as plt
lengths = [len(s) for s in X_train_seq]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(MAX_LEN, color='red', linewidth=2, linestyle='--', label=f'MAX_LEN={MAX_LEN}')
plt.axvline(np.percentile(lengths, 95), color='orange', linewidth=2,
            linestyle='--', label=f'95th percentile={int(np.percentile(lengths,95))}')
plt.xlabel('文章词数')
plt.ylabel('频次')
plt.title('训练集文章长度分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'平均长度：{np.mean(lengths):.0f} 词，最长：{max(lengths)} 词，95% 分位数：{int(np.percentile(lengths,95))} 词')

## 9.5 Embedding 层在 Keras 里的写法

```python
from tensorflow.keras.layers import Embedding

Embedding(
    input_dim=VOCAB_SIZE,   # 词表大小（20,000）
    output_dim=EMBED_DIM,   # 每个词的向量维度（128）
    input_length=MAX_LEN,   # 序列长度（100）
)
```

输入：`(batch_size, 100)` — 100 个整数
输出：`(batch_size, 100, 128)` — 100 个词，每个词 128 维向量

Embedding 层的权重（词表大小 × 嵌入维度 = 20000×128 = 2,560,000 个参数）会在训练中自动更新。

**下一章** 我们将在 CNN 里实际使用这个 Embedding 层。

## 总结

| 概念 | 含义 |
|------|------|
| Embedding | 把整数（词编号）映射到低维稠密向量 |
| Word2Vec | 利用上下文预测训练词向量的经典方法 |
| Tokenizer | 建立词表，把文章转为整数序列 |
| pad_sequences | 把变长序列统一为固定长度 |
| VOCAB_SIZE | 词表大小（保留最高频的 N 个词）|
| MAX_LEN | 序列截断/补齐长度 |
| EMBED_DIM | 每个词的向量维度 |

**下一章 →** Chapter 10：第三个分类器——卷积神经网络（CNN）